# BatchNorm + Dropout Convolutional Neural Network

In [1]:
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import wandb

In [2]:
wandb.login()

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [4]:
DATA = Path.cwd().parent / "data"
OxfordIIITPet(root=DATA, download=True)
PET = DATA / "oxford-iiit-pet"

100%|██████████| 792M/792M [00:21<00:00, 37.2MB/s] 
100%|██████████| 19.2M/19.2M [00:01<00:00, 17.5MB/s]


In [5]:
train_df = pd.read_csv(PET / "annotations" / "trainval.txt", sep=" ", header=None,
                       usecols=[0, 2], names=["image", "species"])

train_df["label"] = train_df["species"] - 1   # 0 = cat, 1 = dog
train_df["breed"] = train_df["image"].str.rsplit("_", n=1).str[0]
train_df["path"] = train_df["image"].apply(lambda name: PET / "images" / f"{name}.jpg")

print(train_df.shape)
train_df.head()

(3680, 5)


,image,species,label,breed,path
0,Abyssinian_100,1,0,Abyssinian,/kaggle/data/oxford-iiit-pet/images/Abyssinian...
1,Abyssinian_101,1,0,Abyssinian,/kaggle/data/oxford-iiit-pet/images/Abyssinian...
2,Abyssinian_102,1,0,Abyssinian,/kaggle/data/oxford-iiit-pet/images/Abyssinian...
3,Abyssinian_103,1,0,Abyssinian,/kaggle/data/oxford-iiit-pet/images/Abyssinian...
4,Abyssinian_104,1,0,Abyssinian,/kaggle/data/oxford-iiit-pet/images/Abyssinian...


# Train:Validation Split

In [6]:
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["breed"]
)

print("Train:", train_data.shape)
print("Validation:", val_data.shape)

Train: (2944, 5)
Validation: (736, 5)


In [7]:
train_data.head()

,image,species,label,breed,path
3399,shiba_inu_166,2,1,shiba_inu,/kaggle/data/oxford-iiit-pet/images/shiba_inu_...
2499,english_setter_159,2,1,english_setter,/kaggle/data/oxford-iiit-pet/images/english_se...
579,Egyptian_Mau_128,1,0,Egyptian_Mau,/kaggle/data/oxford-iiit-pet/images/Egyptian_M...
1742,staffordshire_bull_terrier_143,2,1,staffordshire_bull_terrier,/kaggle/data/oxford-iiit-pet/images/staffordsh...
3376,scottish_terrier_187,2,1,scottish_terrier,/kaggle/data/oxford-iiit-pet/images/scottish_t...


In [8]:
val_data.head()

,image,species,label,breed,path
261,Bengal_110,1,0,Bengal,/kaggle/data/oxford-iiit-pet/images/Bengal_110...
1271,pug_124,2,1,pug,/kaggle/data/oxford-iiit-pet/images/pug_124.jpg
1881,Abyssinian_181,1,0,Abyssinian,/kaggle/data/oxford-iiit-pet/images/Abyssinian...
3239,saint_bernard_153,2,1,saint_bernard,/kaggle/data/oxford-iiit-pet/images/saint_bern...
2270,boxer_170,2,1,boxer,/kaggle/data/oxford-iiit-pet/images/boxer_170.jpg


# Image Transformation

In [9]:
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
])

# Datasets

In [10]:
class PetDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image = Image.open(row["path"]).convert("RGB")
        image = self.transform(image)

        label = int(row["label"])
        label = torch.tensor(label, dtype=torch.long)

        return image, label

In [11]:
train_dataset = PetDataset(train_data, transform)
val_dataset = PetDataset(val_data, transform)

image, label = train_dataset[0]

print("Image shape:", image.shape)
print("Label:", label)
print("Min pixel:", image.min().item())
print("Max pixel:", image.max().item())

Image shape: torch.Size([3, 128, 128])
Label: tensor(1)
Min pixel: 0.01568627543747425
Max pixel: 0.9176470637321472


# Data Loaders

In [12]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

In [13]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([64, 3, 128, 128])
torch.Size([64])


# BatchNormDropoutCNN

In [14]:
class BatchNormDropoutCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256*8*8, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [15]:
model = BatchNormDropoutCNN().to(device)

images, labels = next(iter(train_loader))
images = images.to(device)

outputs = model(images)

print("input shape:", images.shape)
print("output shape:", outputs.shape)

input shape: torch.Size([64, 3, 128, 128])
output shape: torch.Size([64, 2])


In [16]:
num_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {num_params}")

parameters: 4584450


In [17]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

outputs = model(images)
loss = criterion(outputs, labels)

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Loss:", loss.item())

Loss: 0.8124517798423767


# WandB setup

In [18]:
run = wandb.init(
    entity="nestor_dzadzamia-free-university-of-tbilisi-",
    project="pet-classifier-cnn",
    name="batchnorm-dropout-cnn",
    config={
        "architecture": "BatchNormDropoutCNN",
        "dataset": "Oxford-IIIT Pet (cat vs dog)",
        "image_size": 128,
        "epochs": 20,
        "batch_size": 64,
        "learning_rate": 1e-3,
        "optimizer": "Adam",
        "num_conv_layers": 4,
        "filters": "32->64->128->256",
        "fc_units": 256,
        "dropout": 0.5,
        "batch_norm": True,
        "num_params": num_params,
    }
)

# Training Loop

In [19]:
NUM_EPOCHS = 20

model = BatchNormDropoutCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=run.config.learning_rate)

best_f1 = 0.0

for epoch in range(NUM_EPOCHS):
    model.train() # set flag to train (anu batchnorm+dropout working)
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward() # gradientis datvla - backpropagation
        optimizer.step() # weight adjustment

        train_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += images.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    model.eval() # set flag to predict mode
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad(): # stop gradient tracking sichqaristvis
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += images.size(0)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_f1": val_f1,
    })

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "batchnorm_dropout_cnn.pt")
        print("  saved best model")

# Artifact

In [20]:
artifact = wandb.Artifact("batchnorm_dropout_cnn", type="model")
artifact.add_file("batchnorm_dropout_cnn.pt")
run.log_artifact(artifact)

<Artifact batchnorm_dropout_cnn>

In [21]:
wandb.finish()

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▂▃▃▃▄▄▅▅▅▆▆▆▇▆▇▇▇██
train_loss,█▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁
val_acc,▁▂▂▂▂▃▄▃▄▄▅▆▄▇▅▄▆█▅▅
val_f1,▁▄▄▃▃▄▅▃▇▆▆▇▇▇▆▇▇█▇▆
val_loss,▆▅▅▅▅▄▃▇▅▃▂▂▆▂▄▅▄▁▆█
epoch,20
train_acc,0.8502
train_loss,0.31055
val_acc,0.75815
val_f1,0.65961
